# LYRA-Lite — 01 · Dataset & Cohort

**The corpus.** Provenance, loading the cached artifact, and the cohort description that becomes
**Table 1**. In NLP terms this is the *corpus + label* the models consume; the biology is the
substrate, described once here and then it recedes. Reads the cached dataset (built by
`scripts/train.py`); does not recompute preprocessing.

## 1. Data provenance

- **Source:** Single-cell Pediatric Cancer Atlas (ScPCA), project **`SCPCP000008`** (pediatric B-ALL).
- **Modality:** scRNA-seq, `*_processed_rna.h5ad`; `X` = **logcounts** (ScPCA deconvolution normalization).
- **Preprocessing (Hybrid):** reuse ScPCA normalization, then **re-select top-2000 HVGs on the concatenated cohort** (per-file HVG flags are per-sample).
- **Label:** `submitter_celltype_annotation` — `Blast` -> 1; `{T_NK, Monocyte, Plasmablast}` -> 0; `Submitter-excluded` dropped (expert annotation; mild label-noise caveat).
- **Split:** patient-level (`participant_id`), 70/15/15, `GroupShuffleSplit`.

In [8]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np, pandas as pd
import figures                      # local module: all long cohort/table code lives here

# Cached corpus + provenance manifest + per-patient metadata sidecar (built once, cached to CSV).
data = figures.load_cohort()
X, y, groups, genes = data.X, data.y, data.groups, data.genes
manifest, meta, n_hvgs = data.manifest, data.meta, data.n_hvgs
meta.head()

X: (480584, 2000) | genes: 2000
cells: 480584 | blast rate: 0.869 | patients: 84
metadata: 99 patients | diagnoses: 4 | age missing: 0


,diagnosis,subdiagnosis,age,sex,tissue_location,disease_timing
participant_id,,,,,,
SJTALL005006,Early T-cell precursor T-cell acute lymphoblas...,BCL11B-R,10.54,M,Bone marrow,Initial diagnosis
SJALL067628,Early T-cell precursor T-cell acute lymphoblas...,BCL11B-R,0.00,M,Peripheral blood,Initial diagnosis
SJALL067630,Early T-cell precursor T-cell acute lymphoblas...,BCL11B-R,0.00,M,Peripheral blood,Initial diagnosis
SJALL067672,Early T-cell precursor T-cell acute lymphoblas...,BCL11B-R,0.00,F,Bone marrow,Initial diagnosis
SJALL067673,Early T-cell precursor T-cell acute lymphoblas...,BCL11B-R,0.00,M,Bone marrow,Initial diagnosis


## 2. Cohort summary (-> Table 1)

In [3]:
cohort = figures.cohort_summary(groups, y)
cohort.head(20)

patients: 84 | blast fraction range: 0.322 - 0.994


,n_cells,n_blast,n_normal,blast_frac
patient,,,,
SJMLL006,4291,4267.0,24.0,0.994
SJBALL081,7631,7513.0,118.0,0.985
SJHYPO021982,7861,7745.0,116.0,0.985
SJBALL031168,5902,5800.0,102.0,0.983
SJBALL104,6072,5923.0,149.0,0.975
SJBALL030379,4551,4410.0,141.0,0.969
SJALL040119,8461,8201.0,260.0,0.969
SJALL040099,4896,4733.0,163.0,0.967
SJBALL030040,9044,8737.0,307.0,0.966


## 3. Label definition & caveat

`Blast` = 1, `{T_NK, Monocyte, Plasmablast}` = 0, `Submitter-excluded` dropped. Annotation-derived labels -> **mild label noise** (a stated limitation; motivates noisy-label robustness). Dropping `Submitter-excluded` concentrates the labelled remainder, which is why cohort-level blast prevalence reads high.

## 4. Patient-level split (leakage-free)

Blasts differ by patient/subtype, so a cell-level split would leak patient identity and inflate
metrics. The split is patient-disjoint and deterministic given `cfg.seed`; the held-out **test =
unseen patients**. In NLP terms the patient is the **group** in a group-level split — the source of
the distribution-shift stress the models are evaluated under, and the reason cross-seed variance is
dominated by *which* patients are held out.

In [4]:
# leakage-free patient-level split sizes (mirrors scripts/train.py; deterministic given seed)
figures.cohort_split_report(X, y, groups, seed=manifest["params"].get("seed", 42))

train: 349424 cells | 58 patients | blast 0.862
val  :  73112 cells | 13 patients | blast 0.895
test :  58048 cells | 13 patients | blast 0.877


## 5. Cohort 

In [10]:
per_patient = cohort.join(meta)
tex, preview_df = figures.build_table1(cohort, per_patient, n_hvgs)   # writes tables/table1_cohort.tex
preview_df.style.hide(axis="index")

saved -> tables/table1_cohort.tex


Characteristic,Value
Patients (n),84
Total cells (n),"480,584"
Blast cells (n (%)),"417,600 (87%)"
Normal cells (n (%)),"62,984 (13%)"
Cells per patient (median [range]),5533 [973--14659]
Genes per Cell(2000 HVGS) (n),"2,000"
Blast prevalence (%) (median [range]),89 [32--99]
Age at diagnosis (years) (median [range]),7.4 [0.6--18.8]
Sex (n (%)),
M,50 (60%)
